# v2 · Notebook 03 — Avaliação

Compara, em **todas as 6.980 perguntas do dev small** (como nos artigos) e nas **mesmas candidatas do BM25 oficial**:

| Sistema | O que é |
|---|---|
| BM25 | Ordem do run oficial (1º estágio, sem reranqueamento) |
| CE pronto | `cross-encoder/mmarco-mMiniLMv2-L12-H384-v1` (o da v1) |
| CE nosso | O modelo treinado no notebook 02 (+ snapshots para a curva de aprendizado) |

Correções de avaliação da v1 incluídas: teto (Recall@K do BM25), **intervalos de confiança por bootstrap**,
**testes pareados** (Wilcoxon + correção de Holm), **vitórias/empates/derrotas** por pergunta, exemplos de
**derrotas** além de vitórias, e **latência com aquecimento**, p50/p95, na mesma GPU para todos os modelos.
Cada par é pontuado **uma vez** (top-100) e os K menores são fatias disso, já que a pontuação é *pointwise*.

In [ ]:
!pip install -q ftfy==6.3.1   # só para registrar a versão; os textos já vêm corrigidos do notebook 01

In [ ]:
# ==============================================================================
# Google Drive: tudo o que precisa sobreviver a uma desconexão fica aqui
# ==============================================================================
import os
from google.colab import drive
drive.mount("/content/drive")

BASE = "/content/drive/MyDrive/AssignmentPLN_v2"
DATA = f"{BASE}/data"          # saída do notebook 01
CKPT = f"{BASE}/checkpoints"   # saída do notebook 02
RES  = f"{BASE}/results"       # saída do notebook 03
for d in (DATA, CKPT, RES):
    os.makedirs(d, exist_ok=True)

SEED = 42

In [ ]:
import json, time, glob, random, hashlib
import numpy as np
import pandas as pd
import torch
from scipy.stats import wilcoxon
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import transformers

assert torch.cuda.is_available(), "Ative a GPU T4"
DEVICE = "cuda"

# ---------------- Configuração ----------------
READY_MODEL   = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
BASE_MODEL    = "nreimers/mMiniLMv2-L12-H384-distilled-from-XLMR-Large"
RUN_NAME      = "mminilm_bce_150k_lr2e-05_bs32"   # igual ao RUN_NAME do notebook 02
RUN_DIR       = f"{CKPT}/{RUN_NAME}"
K_VALUES      = [10, 25, 50, 100]
MAX_LEN       = 512
FP16_INFER    = True      # aplicado igualmente a todos os modelos
EVAL_SNAPSHOTS = True     # curva de aprendizado
N_SNAPSHOT_QUERIES = 1000 # snapshots avaliados numa subamostra fixa (a curva compara com BM25/CE pronto na MESMA subamostra)
N_BOOT        = 10_000
N_LAT_QUERIES = 100

dev  = pd.read_parquet(f"{DATA}/dev_queries.parquet")
P    = dict(pd.read_parquet(f"{DATA}/dev_passages.parquet").itertuples(index=False, name=None))
meta = json.load(open(f"{DATA}/meta.json"))
print(f"{len(dev)} perguntas dev | {len(P):,} passagens | profundidade {len(dev.candidates[0])}")
print("BM25 Recall@K (teto):", meta["bm25_recall"])

## 1. Pontuar as candidatas (uma vez por modelo, com cache no Drive)

In [ ]:
def carregar(nome):
    if nome == "CE pronto":
        tok = AutoTokenizer.from_pretrained(READY_MODEL)
        m = AutoModelForSequenceClassification.from_pretrained(READY_MODEL)
    elif nome == "CE nosso":
        tok = AutoTokenizer.from_pretrained(f"{RUN_DIR}/final")
        m = AutoModelForSequenceClassification.from_pretrained(f"{RUN_DIR}/final")
    else:   # snapshot: "snap_step{N}"
        tok = AutoTokenizer.from_pretrained(BASE_MODEL)
        m = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=1)
        sd = torch.load(f"{RUN_DIR}/{nome}.pt", map_location="cpu")
        m.load_state_dict({k: v.float() for k, v in sd.items()})
    return tok, m.float().to(DEVICE).eval()   # fp32 para todos (o checkpoint base vem em fp16); o autocast decide a precisão

@torch.no_grad()
def pontuar_query(tok, m, query, pids):
    enc = tok([query] * len(pids), [P[p] for p in pids], truncation=True, max_length=MAX_LEN,
              padding=True, return_tensors="pt").to(DEVICE)
    with torch.autocast("cuda", dtype=torch.float16, enabled=FP16_INFER):
        return m(**enc).logits.squeeze(-1).float().cpu().numpy()

def pontuar_todos(nome, idx):
    """Pontua as candidatas das perguntas dev.iloc[idx]. Cache no Drive por modelo + conjunto exato de perguntas,
    então uma queda de sessão só perde o modelo em andamento."""
    tag = "CE_pronto" if nome == "CE pronto" else f"{RUN_NAME}_{nome.replace(' ', '_')}"
    chave = hashlib.md5(",".join(dev.qid.iloc[idx]).encode()).hexdigest()[:8]
    cache = f"{RES}/scores_{tag}_{len(idx)}q_{chave}.npy"
    if os.path.exists(cache):
        out = list(np.load(cache, allow_pickle=True)); assert len(out) == len(idx); return out
    tok, m = carregar(nome)
    t0 = time.time()
    out = [pontuar_query(tok, m, r.query, list(r.candidates)) for r in dev.iloc[idx].itertuples()]
    print(f"  {nome}: {time.time()-t0:.0f}s")
    np.save(cache, np.array(out, dtype=object), allow_pickle=True)
    del m; torch.cuda.empty_cache()
    return out

ALL = np.arange(len(dev))
SUB = np.sort(np.random.default_rng(SEED).choice(len(dev), size=min(N_SNAPSHOT_QUERIES, len(dev)), replace=False))
qidx = {"CE pronto": ALL, "CE nosso": ALL}                 # quais perguntas cada modelo pontuou
scores = {n: pontuar_todos(n, ALL) for n in qidx}
curve = json.load(open(f"{RUN_DIR}/curve.json"))
if EVAL_SNAPSHOTS:
    for c in curve:
        nome = c["file"][:-3]
        qidx[nome] = SUB
        scores[nome] = pontuar_todos(nome, SUB)

## 2. Métricas por pergunta, médias com IC 95% e teto

In [ ]:
def metricas(ranked, rel, k=10):
    top = ranked[:k]
    mrr = next((1 / i for i, p in enumerate(top, 1) if p in rel), 0.0)
    dcg = sum(1 / np.log2(i + 1) for i, p in enumerate(top, 1) if p in rel)
    idcg = sum(1 / np.log2(i + 1) for i in range(1, min(len(rel), k) + 1))
    rec = sum(1 for p in top if p in rel) / len(rel)
    return mrr, dcg / idcg, rec

def primeiro_rel(ranked, rel):
    return next((i for i, p in enumerate(ranked, 1) if p in rel), np.inf)

def reranquear(cands, s, K):
    o = np.argsort(-s[:K], kind="stable")     # empates mantêm a ordem do BM25
    return [cands[i] for i in o]

rels  = [set(r) for r in dev.relevant_pids]
cands = [list(c) for c in dev.candidates]
runs, ridx = {"BM25": cands}, {"BM25": ALL}   # nome -> rankings, e a quais perguntas eles se referem
for nome, sc in scores.items():
    for K in K_VALUES:
        runs[f"{nome} (K={K})"] = [reranquear(cands[i], s, K) for i, s in zip(qidx[nome], sc)]
        ridx[f"{nome} (K={K})"] = qidx[nome]
per_q = {n: np.array([metricas(rk, rels[i]) for rk, i in zip(lst, ridx[n])]) for n, lst in runs.items()}
first = {n: np.array([primeiro_rel(rk, rels[i]) for rk, i in zip(lst, ridx[n])]) for n, lst in runs.items()}

def ic95(x, chunk=500):
    """IC 95% por bootstrap, em blocos (10k x 6.980 índices de uma vez pesam ~0,5 GB na RAM do Colab)."""
    rng = np.random.default_rng(SEED)
    b = np.concatenate([x[rng.integers(0, len(x), size=(min(chunk, N_BOOT - s), len(x)))].mean(axis=1)
                        for s in range(0, N_BOOT, chunk)])
    return np.percentile(b, 2.5), np.percentile(b, 97.5)

linhas = []
for nome, arr in per_q.items():
    lo, hi = ic95(arr[:, 0])
    linhas.append({"Sistema": nome, "perguntas": len(arr), "MRR@10": arr[:, 0].mean(), "IC95 MRR@10": f"[{lo:.4f}, {hi:.4f}]",
                   "nDCG@10": arr[:, 1].mean(), "Recall@10": arr[:, 2].mean()})
tab = pd.DataFrame(linhas)
principais = tab[~tab.Sistema.str.startswith("snap_")]
principais.to_csv(f"{RES}/tabela_resultados.csv", index=False)
display(principais.round(4))

teto = pd.DataFrame({"K": list(meta["bm25_recall"].keys()), "BM25 Recall@K (teto)": list(meta["bm25_recall"].values())})
teto.to_csv(f"{RES}/teto_bm25.csv", index=False); display(teto.round(4))

## 3. Significância: testes pareados por pergunta (K = 100)

Wilcoxon pareado sobre o MRR@10 de cada pergunta, mais IC 95% da **diferença** por bootstrap pareado.
Correção de Holm para as 3 comparações.

In [ ]:
K = 100
comparacoes = [("CE nosso", "BM25"), ("CE pronto", "BM25"), ("CE nosso", "CE pronto")]
def nm(s): return s if s == "BM25" else f"{s} (K={K})"

res = []
for a, b in comparacoes:
    xa, xb = per_q[nm(a)][:, 0], per_q[nm(b)][:, 0]
    d = xa - xb
    p = wilcoxon(xa, xb).pvalue if np.any(d != 0) else 1.0
    lo, hi = ic95(d)
    res.append({"A": nm(a), "B": nm(b), "ΔMRR@10 (A−B)": d.mean(), "IC95 Δ": f"[{lo:.4f}, {hi:.4f}]", "p (Wilcoxon)": p})
sig = pd.DataFrame(res)
# Holm
ordem = np.argsort(sig["p (Wilcoxon)"].values)
m = len(sig); p_holm = np.empty(m); acc = 0
for r, i in enumerate(ordem):
    acc = max(acc, min(1.0, (m - r) * sig["p (Wilcoxon)"].values[i])); p_holm[i] = acc
sig["p (Holm)"] = p_holm
sig["significativo (α=0,05)"] = sig["p (Holm)"] < 0.05
sig.to_csv(f"{RES}/significancia.csv", index=False)
display(sig)

## 4. Vitórias, empates e derrotas por pergunta (K = 100)

Compara a posição da **primeira passagem relevante**. *Resgate* = estava fora do top 10 e entrou;
*perda* = estava no top 10 e saiu. (Algumas "derrotas" podem ser falsos negativos: passagens relevantes não rotuladas.)

In [ ]:
wtl = []
for a, b in comparacoes:
    ra, rb = first[nm(a)], first[nm(b)]
    wtl.append({"A": nm(a), "B": nm(b),
                "vitórias A": int((ra < rb).sum()), "empates": int((ra == rb).sum()), "derrotas A": int((ra > rb).sum()),
                "resgates (B>10 → A≤10)": int(((rb > 10) & (ra <= 10)).sum()),
                "perdas (B≤10 → A>10)": int(((rb <= 10) & (ra > 10)).sum())})
wtl = pd.DataFrame(wtl); wtl.to_csv(f"{RES}/vitorias_derrotas.csv", index=False); display(wtl)

def r2(x): return -1 if np.isinf(x) else int(x)
ranks_df = pd.DataFrame({"qid": dev.qid, "query": dev["query"],
                         **{f"rank_{n}": [r2(x) for x in first[n]] for n in ["BM25", nm("CE pronto"), nm("CE nosso")]}})
ranks_df.to_csv(f"{RES}/ranks_por_pergunta.csv", index=False)

## 5. Curva de aprendizado do nosso modelo

In [ ]:
if EVAL_SNAPSHOTS:
    cv = []
    for c in curve:
        arr = per_q[f"{c['file'][:-3]} (K=100)"][:, 0]
        lo, hi = ic95(arr)
        cv.append({**c, "MRR@10 (K=100)": arr.mean(), "ic_lo": lo, "ic_hi": hi})
    cv = pd.DataFrame(cv); cv.to_csv(f"{RES}/curva_aprendizado.csv", index=False)
    display(cv[["pairs_seen", "val_pair_acc", "MRR@10 (K=100)", "ic_lo", "ic_hi"]].round(4))

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.errorbar(cv.pairs_seen, cv["MRR@10 (K=100)"], yerr=[cv["MRR@10 (K=100)"] - cv.ic_lo, cv.ic_hi - cv["MRR@10 (K=100)"]],
                marker="o", capsize=3, label="CE nosso (snapshots)")
    ax.axhline(per_q[nm("CE pronto")][SUB, 0].mean(), ls="--", c="gray", label="CE pronto (mesma subamostra)")
    ax.axhline(per_q["BM25"][SUB, 0].mean(), ls=":", c="black", label="BM25 (mesma subamostra)")
    ax.set_title(f"Curva de aprendizado ({len(SUB)} perguntas dev)")
    ax.set_xscale("log"); ax.set_xlabel("pares de treino vistos"); ax.set_ylabel("MRR@10 (dev, K=100)")
    ax.legend(); ax.grid(alpha=.3); fig.tight_layout()
    fig.savefig(f"{RES}/curva_aprendizado.png", dpi=200); plt.show()

## 6. Latência (com aquecimento, mesma GPU, p50/p95)

Mede só o reranqueador (tokenização + modelo), por pergunta. O 1º estágio vem de um run pré-calculado,
então a latência do BM25 **não** é medida aqui (num sistema real, Pyserini/Lucene responde em milissegundos).

In [ ]:
lat = []
sub = dev.sample(N_LAT_QUERIES, random_state=SEED)
for nome in ["CE pronto", "CE nosso"]:
    tok, m = carregar(nome)
    for r in sub.head(10).itertuples():              # aquecimento
        pontuar_query(tok, m, r.query, list(r.candidates))
    for K in K_VALUES:
        ts = []
        for r in sub.itertuples():
            torch.cuda.synchronize(); t0 = time.perf_counter()
            pontuar_query(tok, m, r.query, list(r.candidates)[:K])
            torch.cuda.synchronize(); ts.append((time.perf_counter() - t0) * 1000)
        lat.append({"Modelo": nome, "K": K, "média ms": np.mean(ts), "p50 ms": np.percentile(ts, 50),
                    "p95 ms": np.percentile(ts, 95), "ms por par": np.mean(ts) / K})
    del m; torch.cuda.empty_cache()
lat = pd.DataFrame(lat); lat.to_csv(f"{RES}/latencia.csv", index=False); display(lat.round(2))

## 7. Exemplos qualitativos: vitórias **e** derrotas do nosso modelo sobre o BM25 (K = 100)

In [ ]:
rb, ro = first["BM25"], first[nm("CE nosso")]
vit = np.where((rb > 10) & (ro == 1))[0]
der = np.where((rb <= 3) & (ro > rb))[0]
rnd = random.Random(SEED)
def mostrar(i, titulo):
    row = dev.iloc[i]
    top1 = runs[nm("CE nosso")][i][0]
    print("=" * 100); print(f"{titulo} | qid {row.qid} | BM25: {r2(rb[i])} -> CE nosso: {r2(ro[i])} | CE pronto: {r2(first[nm('CE pronto')][i])}")
    print("PERGUNTA:", row.query)
    print("\nRELEVANTE (rótulo):", P[row.relevant_pids[0]][:600])
    if top1 not in set(row.relevant_pids):
        print("\nTOP-1 DO CE NOSSO (não rotulada como relevante):", P[top1][:600])
print(f"vitórias fortes (BM25>10 -> 1º): {len(vit)} | derrotas (BM25≤3 e piorou): {len(der)}\n")
for i in rnd.sample(list(vit), min(3, len(vit))): mostrar(i, "VITÓRIA")
for i in rnd.sample(list(der), min(3, len(der))): mostrar(i, "DERROTA")

In [ ]:
env = {"torch": torch.__version__, "transformers": transformers.__version__, "gpu": torch.cuda.get_device_name(0),
       "FP16_INFER": FP16_INFER, "MAX_LEN": MAX_LEN, "RUN_NAME": RUN_NAME, "N_BOOT": N_BOOT, "seed": SEED,
       "data_meta": meta}
json.dump(env, open(f"{RES}/env.json", "w"), indent=2, ensure_ascii=False)
print("Resultados em", RES)